# Testando Algoritmo de RandomForest com GKFold para predição de resistência à Compressão

In [ ]:
from sklearn.metrics import r2_score, root_mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.ensemble import RandomForestRegressor
import matplotlib.pyplot as plt
from google.colab import drive
import os, math
import seaborn as sns
import pandas as pd
import numpy as np

In [6]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
dirpath = '/content/drive/MyDrive/supervised-learning-studies/projeto/resistencia'
filename = 'df_resistencia_RF.pkl'
file_path = os.path.join(dirpath, filename)
os.listdir(dirpath)
df = pd.read_pickle(file_path)
df['target'] = df['Resistencia_Compressao_MPa'].copy()
df.drop(inplace=True, columns=['Resistencia_Compressao_MPa'])
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 586 entries, 3 to 598
Data columns (total 16 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Agua_kg_m3                  586 non-null    float64
 1   Usa_SP                      586 non-null    bool   
 2   Cluster_ID                  586 non-null    int64  
 3   Massa_Esp_Areia_kg_m3       586 non-null    float64
 4   Massa_Esp_Brita_kg_m3       586 non-null    float64
 5   vol_brita                   586 non-null    float64
 6   taxa_sp_aglomerante         586 non-null    float64
 7   range_granulometrico        586 non-null    float64
 8   parametro_feret             586 non-null    float64
 9   feret_corrigido_contorno    586 non-null    float64
 10  pow_interacao_tempo         586 non-null    float64
 11  hasselman_resiliencia_agua  586 non-null    float64
 12  aci_estimativa_final_MPa    586 non-null    float64
 13  fib_estimativa_base         586 non-null

In [8]:
n_groups = df['Autores/ano'].nunique()
print(f"Número de grupos únicos (Autores/ano): {n_groups}")
gkf = GroupKFold(n_splits=n_groups)

Número de grupos únicos (Autores/ano): 18


In [14]:
hyper_params = {
    'min_samples_split': 2,
    'min_samples_leaf': 1,
    'max_features': 'sqrt',
    'random_state': 42,
    'n_jobs': -1,
    'bootstrap': True,
    'oob_score': True,
    'warm_start': False
}

In [ ]:
%%time
X = df.drop(columns=['target', 'Autores/ano'])
y = df['target']
groups = df['Autores/ano']

all_preds = []
all_true = []
all_groups = []
results = []
importancias_xgb = []

for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups)):
    
    model = RandomForestRegressor(**hyper_params)

    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    train_errors = []
    val_errors = []
    
    # Simulando a adição de árvores (de 1 a 100, por exemplo)
    max_trees = 100
    for i in range(1, max_trees + 1):
        model.n_estimators = i
        model.fit(X_train, y_train) # Com warm_start, ele só treina a "nova" árvore
        
        # Calculando o erro a cada nova árvore adicionada
        y_train_pred = model.predict(X_train)
        y_val_pred = model.predict(X_val)

        all_preds.extend(y_val_pred)
        all_true.extend(y_val)
        all_groups.extend(groups.iloc[val_idx])
        
        train_errors.append(root_mean_squared_error(y_train, y_train_pred))
        val_errors.append(root_mean_squared_error(y_val, y_val_pred))
    results.append({
        'fold': fold,
        'train_errors': train_errors,
        'val_errors': val_errors
    })


/usr/local/lib/python3.12/dist-packages/sklearn/ensemble/_forest.py:612: UserWarning: Some inputs do not have OOB scores. This probably means too few trees were used to compute any reliable OOB estimates.
  warn(


NameError: name 'mean_squared_error' is not defined

In [18]:
train_curves = []
val_curves = []

for evals_result in results:

    train_curves.append(
        evals_result['validation_0']['rmse']
    )

    val_curves.append(
        evals_result['validation_1']['rmse']
    )

train_curves = np.array(train_curves)
val_curves = np.array(val_curves)

train_mean = train_curves.mean(axis=0)
train_std = train_curves.std(axis=0)

val_mean = val_curves.mean(axis=0)
val_std = val_curves.std(axis=0)

iterations = range(len(train_mean))

plt.figure(figsize=(10, 6))

plt.plot(iterations, train_mean, label='Treino')
plt.plot(iterations, val_mean, label='Validação')

plt.fill_between(
    iterations,
    train_mean - train_std,
    train_mean + train_std,
    alpha=0.2
)

plt.fill_between(
    iterations,
    val_mean - val_std,
    val_mean + val_std,
    alpha=0.2
)

plt.xlabel('Iteração')
plt.ylabel('RMSE')
plt.title('Curva Média de Aprendizado')
plt.legend()
plt.grid(alpha=0.3)

plt.show()

/tmp/ipykernel_9807/3716801628.py:17: RuntimeWarning: Mean of empty slice.
  train_mean = train_curves.mean(axis=0)
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:218: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:175: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:210: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/tmp/ipykernel_9807/3716801628.py:20: RuntimeWarning: Mean of empty slice.
  val_mean = val_curves.mean(axis=0)


TypeError: object of type 'numpy.float64' has no len()

In [ ]:
# Estatísticas finais
train_final = np.mean(train_curves, axis=0)[-1]
val_final = np.mean(val_curves, axis=0)[-1]
overfitting_gap = val_final - train_final

print("\n" + "="*60)
print("📊 ANÁLISE DE OVERFITTING E PERFORMANCE")
print("="*60)
print(f"✅ RMSE Treino (final):      {train_final:.6f}")
print(f"⚠️  RMSE Validação (final):   {val_final:.6f}")
print(f"📈 Gap (Val - Train):        {overfitting_gap:.6f}")
print("="*60)

In [ ]:
debug_df = pd.DataFrame({
    'y_true': np.array(all_true).ravel(),
    'y_pred': np.array(all_preds).ravel(),
    'group': np.array(all_groups).ravel()
})

debug_df['residual'] = (
    debug_df['y_true'] - debug_df['y_pred']
)

In [ ]:
plt.figure(figsize=(8, 6))

sns.histplot(
    debug_df['residual'],
    bins=30,
    kde=True,
    color='purple',
    alpha=0.6
)

plt.title("Distribuição dos Resíduos")
plt.xlabel("Resíduo (Ground Truth - Predição)")
plt.ylabel("Frequência")

plt.grid(True, alpha=0.3)

plt.show()

In [ ]:
# Ordenando pelo valor real para a linha de predição não virar um "zigue-zague"
debug_df = debug_df.sort_values(by="y_true").reset_index(drop=True)
indices = range(len(debug_df))

plt.figure(figsize=(12, 6))

# 1. Linha das Predições (conectando os pontos preditos)
plt.plot(indices, debug_df["y_pred"], color='blue', label="Linha de Predição", 
         alpha=0.7, linewidth=1.5, marker='o', markersize=4)

# 2. Pontos dos Valores Reais (Ground Truth)
plt.plot(indices, debug_df["y_true"], color='red', label="Valores Reais", 
            alpha=0.5, linewidth=1.5, marker='s', markersize=4)

plt.title("Linha de Predição vs Valores Reais (Ordenados)")
plt.xlabel("Amostras (Ordenadas pelo valor real)")
plt.ylabel("Resistência")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
r2 = r2_score(debug_df["y_true"], debug_df["y_pred"])
rmse = root_mean_squared_error(debug_df["y_true"], debug_df["y_pred"])
mae = mean_absolute_error(debug_df["y_true"], debug_df["y_pred"])
mape = mean_absolute_percentage_error(debug_df["y_true"], debug_df["y_pred"])

print(f"root_mean_squared_error: {rmse:.4f}")
print(f"mean_absolute_error: {mae:.4f}")
print(f"mean_absolute_percentage_error: {mape:.4f}")
print(f"R² Score: {(r2*100):.4f}")

In [ ]:
# 1. Extraindo as importâncias direto do modelo XGBoost treinado
# (Assumindo que o seu modelo se chame 'model_xgb')
importancias_xgb = model.feature_importances_

# 2. Criando o DataFrame para organizar os dados
df_importancia_xgb = pd.DataFrame({
    'Feature': X_train.columns,
    'Importancia': importancias_xgb
})

# Ordenando da maior para a menor importância
df_importancia_xgb = df_importancia_xgb.sort_values(by='Importancia', ascending=True)

# 3. Plotando o Gráfico
plt.figure(figsize=(10, 8))

# Criando uma paleta de cores (usando o 'viridis' para diferenciar do gráfico anterior)
cores = sns.color_palette('viridis', n_colors=len(df_importancia_xgb))

# Plotando as barras horizontais
plt.barh(
    y=df_importancia_xgb['Feature'], 
    width=df_importancia_xgb['Importancia'],
    color=cores,
    edgecolor='none'
)

plt.title('Importância das Variáveis (Ganho de Informação) - XGBoost', fontsize=14)
plt.xlabel('Importância Relativa (Impacto na quebra dos nós)', fontsize=12)
plt.ylabel('Variáveis', fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()